# Étude de Cas: FIFA World Cup 2026 pipeline development

This notebook performs:
- Creation of a dedicated Volume
- Secure loading of API key from `.env`  
- Import of custom module `api_client.py`  
- Extraction of fixtures from API‑Football  
- Grouped extraction to avoid rate limits  
- Incremental JSON landing in Volumes  
- Ready for COPY INTO → Bronze

In [0]:
%sql
-- Step 0: Volume Creation
CREATE VOLUME IF NOT EXISTS workspace.default.api_football_pipeline;

In [0]:
# Step 0.1: Subdirs Creation
base = "/Volumes/workspace/default/api_football_pipeline"

dbutils.fs.mkdirs(f"{base}/input")
dbutils.fs.mkdirs(f"{base}/bronze")
dbutils.fs.mkdirs(f"{base}/silver")
dbutils.fs.mkdirs(f"{base}/gold")
dbutils.fs.mkdirs(f"{base}/checkpoints")
dbutils.fs.mkdirs(f"{base}/libs")


True

In [0]:
# Step 0.2: Package installation
####requirements.txt must be in {base}
req_path = f"{base}/requirements.txt"
%pip install -r $req_path

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Step 1: Import Custom Library for API Extraction
import sys
sys.path.append(f"{base}/libs")

import api_client

# Reload if you make changes in the library
import importlib
importlib.reload(api_client)

# Use the class from the module (this WILL reflect changes)
APIFootballClient = api_client.APIFootballClient


In [0]:
# Step 2: Instantiate the API Client 

# For this it would be nice to know which attributes are accepted in the __init__
import inspect
print(inspect.getfullargspec(APIFootballClient))

FullArgSpec(args=['self', 'env_path'], varargs=None, varkw=None, defaults=(None,), kwonlyargs=[], kwonlydefaults=None, annotations={'env_path': <class 'str'>})


In [0]:
client = APIFootballClient(
    env_path=f"{base}/.env"
)

In [0]:
##### This block will serve to define the 48 teams we will be working with
teams = [
    "Argentina", "Brazil", "Uruguay", "Colombia", "Ecuador", "Peru", "Chile", "Paraguay", "Venezuela", "Bolivia",
    "Mexico", "USA", "Canada", "Costa Rica", "Panama", "Honduras", "El Salvador", "Guatemala", "Haiti", "Jamaica",
    "Trinidad and Tobago", "Curaçao", "Suriname", "Nicaragua", "Dominican Republic", "Puerto Rico", "Cuba",
    "England", "France", "Germany", "Spain", "Portugal", "Italy", "Netherlands", "Belgium", "Croatia", "Switzerland",
    "Serbia", "Denmark", "Sweden", "Norway", "Poland", "Austria", "Czech Republic", "Ukraine", "Turkey", "Greece"
]

In [0]:
# Step 3: Group Teams to Avoid API Rate Limits
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

team_groups = list(chunk_list(teams, 4))
team_groups

[['Argentina', 'Brazil', 'Uruguay', 'Colombia'],
 ['Ecuador', 'Peru', 'Chile', 'Paraguay'],
 ['Venezuela', 'Bolivia', 'Mexico', 'USA'],
 ['Canada', 'Costa Rica', 'Panama', 'Honduras'],
 ['El Salvador', 'Guatemala', 'Haiti', 'Jamaica'],
 ['Trinidad and Tobago', 'Curaçao', 'Suriname', 'Nicaragua'],
 ['Dominican Republic', 'Puerto Rico', 'Cuba', 'England'],
 ['France', 'Germany', 'Spain', 'Portugal'],
 ['Italy', 'Netherlands', 'Belgium', 'Croatia'],
 ['Switzerland', 'Serbia', 'Denmark', 'Sweden'],
 ['Norway', 'Poland', 'Austria', 'Czech Republic'],
 ['Ukraine', 'Turkey', 'Greece']]

In [0]:
# Step 4: Extract Fixtures per Group
import json
from datetime import datetime

group = team_groups[0]

all_fixtures = []

for team in group:
    print(f"Extrayendo fixtures de: {team}")
    team_id = client.get_national_team_id(team)

    if not team_id:
        print(f"⚠️ No se encontró ID para {team}")
        continue

    fixtures = client.get_fixtures_by_team_id(team_id, start_year=2023, end_year=2026)
    all_fixtures.extend(fixtures)


Extrayendo fixtures de: Argentina
Error en la solicitud: HTTPSConnectionPool(host='v3.football.api-sports.io', port=443): Max retries exceeded with url: /teams?search=Argentina (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0xffcb9c294c20>: Failed to resolve 'v3.football.api-sports.io' ([Errno -3] Temporary failure in name resolution)"))
⚠️ No se encontró ID para Argentina
Extrayendo fixtures de: Brazil
Error en la solicitud: HTTPSConnectionPool(host='v3.football.api-sports.io', port=443): Max retries exceeded with url: /teams?search=Brazil (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0xffcb8046ef90>: Failed to resolve 'v3.football.api-sports.io' ([Errno -3] Temporary failure in name resolution)"))
⚠️ No se encontró ID para Brazil
Extrayendo fixtures de: Uruguay
Error en la solicitud: HTTPSConnectionPool(host='v3.football.api-sports.io', port=443): Max retries exceeded with url: /teams?search=Uruguay (Caused by NameResolut

The code above failed due to the use of Databricks Free Edition. Naturally, this edition doesn't have egress, network permissions, or compute with Internet.

The alternative is going to be a local extraction and storage in the volume manually

In [0]:
# Step 5: JSON Store in Landing (input)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

dbutils.fs.put(
    f"{base}/input/fixtures_group_{timestamp}.json",
    json.dumps(all_fixtures),
    overwrite=True
)
